# Phase 2: Data Warehouse Design (Star Schema)

## Executive Summary
In Phase 1, raw Olist e-commerce data was successfully ingested into our Google BigQuery environment (`olist_raw`). 
This notebook documents Phase 2, where we transform that raw data into a structured **Star Schema** within our Data Warehouse (`olist_dwh`). 

This architectural shift ensures our data is highly organized, strictly typed, and optimized for downstream business intelligence and analytics.

### Project Milestones Achieved:
1. **Defined the Conceptual Architecture:** Mapped raw tables to Fact and Dimension tables.
2. **Established Data Governance:** Shifted from manual database DDL (Create Table scripts) to automated, version-controlled ELT using **dbt (Data Build Tool)**.
3. **Deployed the Data Mart:** Successfully built the core schema in BigQuery.


## 1. Visualizing the Architecture
A Star Schema separates business process events (Facts) from descriptive attributes (Dimensions). Below is the Entity-Relationship Diagram (ERD) defining our target architecture.


```mermaid
erDiagram
    fact_orders {
        string order_item_sk PK
        string order_key 
        string product_key FK
        string seller_key FK
        string customer_key FK
        string date_key FK
        float price
        float freight_value
    }
    dim_date {
        string date_key PK
        date full_date
        int year
        int month
        int quarter
        int day_of_week
        boolean is_weekend
    }
    dim_customers {
        string customer_key PK
        string customer_unique_id
        string customer_city
        string customer_state
    }
    dim_products {
        string product_key PK
        string category_name
        float weight_g
    }
    dim_sellers {
        string seller_key PK
        string seller_city
        string seller_state
        string seller_zip_code_prefix
    }

    dim_date ||--o{ fact_orders : "1 to N"
    dim_customers ||--o{ fact_orders : "1 to N"
    dim_products ||--o{ fact_orders : "1 to N"
    dim_sellers ||--o{ fact_orders : "1 to N"
```

## 2. Schema Design Justification
When building a data system, the architecture must support the end-users (data analysts and business teams). We selected a Star Schema for the following technical and business reasons:

* **Query Efficiency:** By pre-joining and flattening complex relational data into isolated dimensions, analysts can perform fast aggregations (e.g., total sales by month) without writing complex, multi-table joins.
* **Data Integrity:** Surrogate and Primary Keys across our dimension tables (`dim_customers`, `dim_products`, `dim_sellers`, `dim_date`) ensure a single source of truth.
* **Scalability:** As the Olist dataset grows, the central `fact_orders` table can scale vertically, while dimension tables handle attribute updates seamlessly.

## 3. Implementation Methodology (dbt)
Rather than executing manual `CREATE TABLE` statements, our team utilized **dbt** to manage our data transformations. 

This approach abstracts the DDL logic. We write purely functional `SELECT` statements (models) in the `marts/` folder, which pull clean data from our `staging/` layer using the `{{ ref() }}` function. dbt then automatically compiles and materializes these models as tables in BigQuery.

In [2]:
# ==========================================
# EXAMPLE: DIMENSION MODEL (dim_customers.sql)
# ==========================================
# Location: dbt_olist/models/marts/dim_customers.sql

sql_dim_customers = """
{{ config(materialized='table') }}

WITH staging_customers AS (
    SELECT * FROM {{ ref('stg_customers') }}
)

SELECT
    customer_id AS customer_key,
    customer_unique_id,
    customer_city,
    customer_state
FROM staging_customers
"""

print("Dimension models successfully map attributes from staging to the Data Warehouse.")

Dimension models successfully map attributes from staging to the Data Warehouse.


In [ ]:
# ==========================================
# EXAMPLE: FACT MODEL (fact_orders.sql)
# ==========================================
# Location: dbt_olist/models/marts/fact_orders.sql

sql_fact_orders = """
{{ config(materialized='table') }}

WITH staging_order_items AS (
    SELECT * FROM {{ ref('stg_order_items') }}
)

SELECT
    order_item_id AS order_item_sk,    
    order_id AS order_key,             
    product_id AS product_key,         
    seller_id AS seller_key,           
    price,
    freight_value
FROM staging_order_items
"""

print("Fact models successfully capture measurable events and foreign keys.")

## 4. Deployment & Testing
To materialize this architecture in our production BigQuery environment, the following execution plan was executed via the terminal:

1. **Test the Connection:** Verified GCP Service Account permissions.
   `dbt debug`
2. **Build the Schema:** Compiled the SQL and materialized the tables in BigQuery.
   `dbt run --select marts`

**Result:** All 5 core tables (`dim_customers`, `dim_products`, `dim_sellers`, `dim_date`, `fact_orders`) were successfully created in the `olist_dwh` dataset. 

## Next Steps (Phase 3 & 4)
With the Star Schema structurally complete, the pipeline is ready for **Data Quality Testing** (e.g., asserting positive prices, validating order statuses) and advanced **Data Analysis** to uncover business insights.